# **<font color='#6edce9'>Validancion de consolidado</font>**
### **Objetivo:** Construir el pipeline de mapeo y limpieza para unificar los datasets anuales de RRHH y migrar la base de datos consolidada a PostgreSQL.

## **<font color='#eda985'> 1. Importación y llamada a función para consolidar los 7 excel (2019 hasta 2027)</font>**

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))  # sube un nivel desde notebooks/ hacia la raíz

from src.minsa_analystics.etl import consolidar_datos

# ============================================================
# CONFIGURACIÓN
# ============================================================
# --- Carga con caché en Parquet, para no re-procesar los 7 Excel cada vez ---
FORZAR_RECARGA = True  # IMPORTANTE⚠️: . cambia a True cuando se modifique algo  etl.py o corrijas datos

ruta_datos = str(Path.cwd().parent / 'data' / 'raw')
ruta_parquet = Path.cwd().parent / 'data' / 'processed' / 'consolidado.parquet'

# ============================================================
# CARGA / GENERACIÓN DEL CONSOLIDADO
# ============================================================
if ruta_parquet.exists() and not FORZAR_RECARGA:
    print("Cargando consolidado desde cache (parquet)...")
    df = pd.read_parquet(ruta_parquet)
else:
    print("Generando consolidado desde los 7 Excel...")
    df = consolidar_datos(str(ruta_datos))

    # Crear carpeta si no existe
    ruta_parquet.parent.mkdir(parents=True, exist_ok=True)

    # Guardar Parquet
    df.to_parquet(ruta_parquet, index=False)

    print(f"\nParquet guardado en:")
    print(ruta_parquet)
    print(f"\n¿Archivo creado?: {ruta_parquet.exists()}")

    if ruta_parquet.exists():
        print(f"Tamaño: {ruta_parquet.stat().st_size / 1024**2:.2f} MB")


print(f"\nDimensiones: {df.shape}")

In [ ]:
# Verificando datos del campo 'es_especialista'
print(df.groupby(['anio_registro', 'es_especialista'], dropna=False).size().unstack(fill_value=0))

## **<font color='#eda985'> 2. Verificando cantidad de registros</font>**

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.1f}'.format)

# Filas por año (deberían coincidir con lo que ya viste arriba)
print('Cantidad de registros por año')
print('='*30)
print(df['anio_registro'].value_counts().sort_index())

In [ ]:
df.head(10)

In [ ]:
df.columns

In [ ]:
# Revisando tipo de dato de cada columna

for col in df.columns:
    tipos = df[col].map(type).unique()
    if len(tipos) > 1:
        print(col, '->', tipos)

## **<font color='#eda985'> 3. Verificando nulos de cada columna</font>**

In [ ]:
# % de nulos por columna, para detectar algo inesperado
print('% Columnas Nulas')
print('='*30)
print(df.isna().mean().sort_values(ascending=False).head(10))

## **<font color='#eda985'> 4. Auditoría y Perfilado de Errores</font>**

### **<font color='#96f499'> 4.1. Verificando y corrigiendo campos Relacionados: id_especialidad, especialidad,condicion_especialidad, es_especialista </font>**

### Reglas de Negocio: Validación de Especialistas
Para validar la consistencia de los datos, la columna `es_especialista` debe cruzarse obligatoriamente con los campos de especialidad (`id_especialidad`, `especialidad`, `condicion_especialidad`) y `la condición laboral`.

##### 1. Cuando `es_especialista` es **"SI"** 🟢
**Regla estricta:** Todos los campos de especialidad deben estar **completamente llenos**.
**Campos obligatorios:**
  * `id_especialidad`
  * `especialidad`
  * `condicion_especialidad`
*Cualquier registro en "SI" con alguno de estos campos vacío se considera un **Error de Consistencia**.*

##### 2. Cuando `es_especialista` es **"NO"** 🟡
* **Regla flexible:** Los campos de especialidad **pueden o no estar llenos** (se permite que tengan datos o que estén vacíos).
* **Excepción por Residencia:** Si el registro tiene `condicion_laboral` = **"residente"**, el valor de `es_especialista` será obligatoriamente **"NO"**, incluso si ya tiene asignada una especialidad en los campos de texto.

---

> 💡 **Nota de Diagnóstico:** Tras analizar el set de datos, los **11 registros** con valor "SI" y especialidad vacía (`False`) representan la inconsistencia más grave, ya que violan la regla estricta del negocio. Los registros en "NO", "No" o "0" que contienen datos pueden justificarse bajo la excepción de médicos residentes.
Usa el código con precaución.
🛠️ Código de Pandas actualizado con la nueva regla
Para que tu script refleje exactamente esta nueva consideración de los residentes, puedes usar este enfoque para aislar únicamente los errores reales (los 11 "SI" vacíos y cualquier "SI" que sea residente por error):
python

In [ ]:
# 1. Filtro estricto: Es "SI" pero le falta alguna de las 3 columnas de especialidad
campos_incompletos = df['id_especialidad'].isna() | df['especialidad'].isna() | df['condicion_especialidad'].isna()
error_si_incompleto = (df['es_especialista'] == 'SI') & campos_incompletos

# 2. Filtro de contradicción: Dice "SI" pero su condición laboral es "residente" (debería ser NO)
error_residente_especialista = (df['es_especialista'] == 'SI') & (df['condicion_laboral'].str.lower() == 'residente')

# Unimos los errores reales del sistema
errores_reales = df[error_si_incompleto | error_residente_especialista]

print(f"Total de registros con errores críticos detectados: {len(errores_reales)}")
print(f'Total regitros indican SI es especialista pero alguno de los 3 campos vacios: {error_si_incompleto.sum()}')
print(f'Total regitros indican SI es especialista pero condicion laboral es Residente: {error_residente_especialista.sum()} ')

In [ ]:
# Verificando campo 'tiene_especialidad' con 'especialidad' NO VACÍO
df['tiene_especialidad'] = df['especialidad'].notna() # es_especialista vs especialidad llena
print(df.groupby(['es_especialista', 'tiene_especialidad'], dropna=False).size())

### **<font color='#96f499'> 4.2. Verificando detalle del campo EDAD </font>**

In [ ]:
# EDAD: rango razonable en todo el consolidado
print('Descripcion EDAD')
print('='*30)
print(df['edad'].describe())

In [ ]:
# Registros con edad mayo a 80 años
df[df['edad'] > 80][['edad', 'anio_registro']].value_counts()